In [2]:
import pandas as pd 

ts_data = pd.read_parquet("../data/transformed/ts_data_2026_01.parquet")
ts_data

,pickup_hour,rides,pickup_location_id
0,2026-01-01 00:00:00,0,1
1,2026-01-01 01:00:00,0,1
2,2026-01-01 02:00:00,1,1
3,2026-01-01 03:00:00,0,1
4,2026-01-01 04:00:00,3,1
...,...,...,...
194923,2026-01-31 19:00:00,0,265
194924,2026-01-31 20:00:00,1,265
194925,2026-01-31 21:00:00,1,265
194926,2026-01-31 22:00:00,6,265


In [3]:
ts_data_one_location = ts_data.loc[ts_data.pickup_location_id == 43, :].reset_index(drop=True)

ts_data_one_location.head(10)

,pickup_hour,rides,pickup_location_id
0,2026-01-01 00:00:00,166,43
1,2026-01-01 01:00:00,99,43
2,2026-01-01 02:00:00,30,43
3,2026-01-01 03:00:00,17,43
4,2026-01-01 04:00:00,6,43
5,2026-01-01 05:00:00,4,43
6,2026-01-01 06:00:00,7,43
7,2026-01-01 07:00:00,8,43
8,2026-01-01 08:00:00,13,43
9,2026-01-01 09:00:00,14,43


In [9]:
def get_cutoff_indices(
    data: pd.DataFrame,
    n_features: int,
    step_size: int
) -> list:
    """
    Generates the indices needed to create sliding windows for a time-series dataset.
    """
    stop_position = len(data) - 1

    # Start from first sub_sequence at index position 0
    subseq_first_idx = 0
    subseq_mid_index = n_features
    subseq_last_idx = n_features + 1

    indices = []
    while subseq_last_idx <= stop_position:
        indices.append((subseq_first_idx, subseq_mid_index, subseq_last_idx))

        subseq_first_idx += step_size
        subseq_mid_index += step_size
        subseq_last_idx += step_size
    
    return indices


In [12]:
n_features = 24
step_size = 1

indices = get_cutoff_indices(
    ts_data_one_location,
    n_features,
    step_size,
)
indices[:6]

[(0, 24, 25), (1, 25, 26), (2, 26, 27), (3, 27, 28), (4, 28, 29), (5, 29, 30)]

In [18]:
import numpy as np

n_samples = len(indices)
x = np.ndarray(shape=(n_samples, n_features), dtype=np.float32)
y = np.ndarray(shape=(n_samples), dtype=np.float32) 

picked_hours = []

for i, idx in enumerate(indices):
    x[i, :] = ts_data_one_location.iloc[idx[0]:idx[1]]['rides'].values
    y[i] = ts_data_one_location.iloc[idx[1]:idx[2]]['rides'].values[0]
    picked_hours.append(ts_data_one_location.iloc[idx[1]]['pickup_hour'])



In [19]:
print(f'{x.shape}')
print(f'{x=}')
print(f'{picked_hours[:5]=}')

(719, 24)
x=array([[166.,  99.,  30., ...,  22.,  17.,   9.],
       [ 99.,  30.,  17., ...,  17.,   9.,   3.],
       [ 30.,  17.,   6., ...,   9.,   3.,   2.],
       ...,
       [ 91.,  58.,  26., ..., 149., 106.,  74.],
       [ 58.,  26.,  21., ..., 106.,  74.,  68.],
       [ 26.,  21.,  18., ...,  74.,  68.,  73.]],
      shape=(719, 24), dtype=float32)
picked_hours[:5]=[Timestamp('2026-01-02 00:00:00'), Timestamp('2026-01-02 01:00:00'), Timestamp('2026-01-02 02:00:00'), Timestamp('2026-01-02 03:00:00'), Timestamp('2026-01-02 04:00:00')]


In [20]:
feature_one_location = pd.DataFrame(
    x,
    columns=[f'rides_previous_{i+1}_hours' for i in reversed(range(n_features))]
)
feature_one_location

,rides_previous_24_hours,rides_previous_23_hours,rides_previous_22_hours,rides_previous_21_hours,rides_previous_20_hours,rides_previous_19_hours,rides_previous_18_hours,rides_previous_17_hours,rides_previous_16_hours,rides_previous_15_hours,...,rides_previous_10_hours,rides_previous_9_hours,rides_previous_8_hours,rides_previous_7_hours,rides_previous_6_hours,rides_previous_5_hours,rides_previous_4_hours,rides_previous_3_hours,rides_previous_2_hours,rides_previous_1_hours
0,166.0,99.0,30.0,17.0,6.0,4.0,7.0,8.0,13.0,14.0,...,87.0,124.0,63.0,93.0,53.0,52.0,27.0,22.0,17.0,9.0
1,99.0,30.0,17.0,6.0,4.0,7.0,8.0,13.0,14.0,41.0,...,124.0,63.0,93.0,53.0,52.0,27.0,22.0,17.0,9.0,3.0
2,30.0,17.0,6.0,4.0,7.0,8.0,13.0,14.0,41.0,53.0,...,63.0,93.0,53.0,52.0,27.0,22.0,17.0,9.0,3.0,2.0
3,17.0,6.0,4.0,7.0,8.0,13.0,14.0,41.0,53.0,61.0,...,93.0,53.0,52.0,27.0,22.0,17.0,9.0,3.0,2.0,2.0
4,6.0,4.0,7.0,8.0,13.0,14.0,41.0,53.0,61.0,120.0,...,53.0,52.0,27.0,22.0,17.0,9.0,3.0,2.0,2.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
714,123.0,112.0,91.0,58.0,26.0,21.0,18.0,5.0,1.0,2.0,...,22.0,44.0,63.0,91.0,98.0,182.0,131.0,164.0,166.0,149.0
715,112.0,91.0,58.0,26.0,21.0,18.0,5.0,1.0,2.0,3.0,...,44.0,63.0,91.0,98.0,182.0,131.0,164.0,166.0,149.0,106.0
716,91.0,58.0,26.0,21.0,18.0,5.0,1.0,2.0,3.0,4.0,...,63.0,91.0,98.0,182.0,131.0,164.0,166.0,149.0,106.0,74.0
717,58.0,26.0,21.0,18.0,5.0,1.0,2.0,3.0,4.0,6.0,...,91.0,98.0,182.0,131.0,164.0,166.0,149.0,106.0,74.0,68.0


In [21]:
taget_one_location = pd.DataFrame(
    y, columns=[f'target_rides_next_hour']
)
taget_one_location


,target_rides_next_hour
0,3.0
1,2.0
2,2.0
3,1.0
4,1.0
...,...
714,106.0
715,74.0
716,68.0
717,73.0


In [27]:
from tqdm import tqdm 
from typing import Tuple

def transform_ts_data_info_feature_and_target(
    ts_data: pd.DataFrame,
    input_seq_len: int,
    step_size: int
) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Slice and Transform data from time-series format into a (feature,target)
    foramt that we use it to train Supervised ML models.
    """
    assert set(ts_data.columns) == {'pickup_hour','rides', 'pickup_location_id'}
    
    location_ids = ts_data['pickup_location_id'].unique()
    features = pd.DataFrame()
    targets = pd.DataFrame()

    for location_id in tqdm(location_ids):

        # keep only ts_data for 'location_id'
        ts_data_one_location = ts_data.loc[
            ts_data.pickup_location_id == location_id,['pickup_hour','rides']
        ]

        # Compute cutoff indices to split dataframe rows
        indices = get_cutoff_indices(
            ts_data_one_location,
            input_seq_len,
            step_size,
        )

         # Slice and transpose data into numpy array for features and target
        n_samples = len(indices)
        x = np.ndarray(shape=(n_samples, input_seq_len), dtype=np.float32)
        y = np.ndarray(shape=(n_samples), dtype=np.float32) 

        pickup_hours = []

        for i, idx in enumerate(indices):
            x[i, :] = ts_data_one_location.iloc[idx[0]:idx[1]]['rides'].values
            y[i] = ts_data_one_location.iloc[idx[1]:idx[2]]['rides'].values[0]
            pickup_hours.append(ts_data_one_location.iloc[idx[1]]['pickup_hour'])

        # numpy -> pandas
        feature_one_location = pd.DataFrame(
            x,
            columns=[f'rides_previous_{i+1}_hours' for i in reversed(range(input_seq_len))]
        )
        feature_one_location['pickup_hour'] = pickup_hours
        feature_one_location['pickup_location_id'] = location_id

        # numpy -> pandas
        target_one_location = pd.DataFrame(
            y, columns=[f'target_rides_next_hour']
        )

        # Concatenate results 
        features = pd.concat([features, feature_one_location])
        targets = pd.concat([targets, target_one_location])
    
    features.reset_index(inplace=True, drop=True)
    targets.reset_index(inplace=True, drop=True)
    
    return features, targets['target_rides_next_hour'] 


In [30]:
featues, targets = transform_ts_data_info_feature_and_target(
    ts_data,
    input_seq_len=24*7*1, # one wekk of history
    step_size=24,
)
print(f'{featues.shape}')
print(f'{targets.shape}')


100%|██████████| 262/262 [00:02<00:00, 117.83it/s]

(6288, 170)
(6288,)
